<a href="https://colab.research.google.com/github/Brandeis-Visual-Analytics/COSI-165b-labs/blob/main/lab4/lab_4_Backprop_Initialization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Backpropagation and Initialization

**REMEMBER TO File > Save a Copy in Drive IF YOU ARE USING COLAB**

This notebook has two parts:

* **Part 1: Backpropagation.** You'll implement the forward and backward passes of a deep neural network, as described in section 7.4 of the book, and check your derivatives against finite differences.
* **Part 2: Initialization.** You'll use the functions you wrote in Part 1 to explore how the initial weights affect the size of the activations and gradients in deep networks, as described in section 7.5 of the book.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Adapted from the notebooks for *Understanding Deep Learning* by Simon J.D. Prince (udlbookmail@gmail.com).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# **Part 1: Backpropagation**

First let's define a neural network.  We'll just choose the weights and biases randomly for now

In [ ]:
# Set seed so we always get the same random numbers
np.random.seed(0)

# Number of hidden layers
K = 5
# Number of neurons per layer
D = 6
# Input layer
D_i = 1
# Output layer
D_o = 1

# Make empty lists
all_weights = [None] * (K+1)
all_biases = [None] * (K+1)

# Create input and output layers
all_weights[0] = np.random.normal(size=(D, D_i))
all_weights[-1] = np.random.normal(size=(D_o, D))
all_biases[0] = np.random.normal(size =(D,1))
all_biases[-1]= np.random.normal(size =(D_o,1))

# Create intermediate layers
for layer in range(1,K):
  all_weights[layer] = np.random.normal(size=(D,D))
  all_biases[layer] = np.random.normal(size=(D,1))

In [ ]:
# Define the Rectified Linear Unit (ReLU) function
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation

Now let's run our random network.  The weight matrices $\boldsymbol\Omega_{0\ldots K}$ are the entries of the list "all_weights" and the biases $\boldsymbol\beta_{0\ldots K}$ are the entries of the list "all_biases"

We know that we will need the preactivations $\mathbf{f}_{0\ldots K}$ and the activations $\mathbf{h}_{1\ldots K}$ for the forward pass of backpropagation, so we'll store and return these as well.


In [ ]:
def compute_network_output(net_input, all_weights, all_biases):

  # Retrieve number of layers
  K = len(all_weights) -1

  # We'll store the pre-activations at each layer in a list "all_f"
  # and the activations in a second list "all_h".
  all_f = [None] * (K+1)
  all_h = [None] * (K+1)

  #For convenience, we'll set
  # all_h[0] to be the input, and all_f[K] will be the output
  all_h[0] = net_input

  # Run through the layers, calculating all_f[0...K-1] and all_h[1...K]
  for layer in range(K):
      # Update preactivations and activations at this layer according to eqn 7.17
      # Remember to use np.matmul for matrix multiplications
      # TODO -- Replace the lines below
      all_f[layer] = all_h[layer]
      all_h[layer+1] = all_f[layer]

  # Compute the output from the last hidden layer
  # TODO -- Replace the line below
  all_f[K] = np.zeros_like(all_biases[-1])

  # Retrieve the output
  net_output = all_f[K]

  return net_output, all_f, all_h

In [ ]:
# Define input
net_input = np.ones((D_i,1)) * 1.2
# Compute network output
net_output, all_f, all_h = compute_network_output(net_input,all_weights, all_biases)
print("True output = %3.3f, Your answer = %3.3f"%(1.907, net_output[0,0]))

Now let's define a loss function.  We'll just use the least squares loss function. We'll also write a function to compute dloss_doutput

In [ ]:
def least_squares_loss(net_output, y):
  return np.sum((net_output-y) * (net_output-y))

def d_loss_d_output(net_output, y):
    return 2*(net_output -y);

In [ ]:
y = np.ones((D_o,1)) * 20.0
loss = least_squares_loss(net_output, y)
print("y = %3.3f Loss = %3.3f"%(y[0,0], loss))

Now let's compute the derivatives of the network.  We already computed the forward pass.  Let's compute the backward pass.

In [ ]:
# We'll need the indicator function
def indicator_function(x):
  x_in = np.array(x)
  x_in[x_in>0] = 1
  x_in[x_in<=0] = 0
  return x_in

# Main backward pass routine
def backward_pass(all_weights, all_biases, all_f, all_h, y):
  # Retrieve number of layers (so this works for networks of any depth -- you'll reuse it in Part 2)
  K = len(all_weights) - 1

  # We'll store the derivatives dl_dweights and dl_dbiases in lists as well
  all_dl_dweights = [None] * (K+1)
  all_dl_dbiases = [None] * (K+1)
  # And we'll store the derivatives of the loss with respect to the activation and preactivations in lists
  all_dl_df = [None] * (K+1)
  all_dl_dh = [None] * (K+1)
  # Again for convenience we'll stick with the convention that all_h[0] is the net input and all_f[k] in the net output

  # Compute derivatives of the loss with respect to the network output
  all_dl_df[K] = np.array(d_loss_d_output(all_f[K],y))

  # Now work backwards through the network
  for layer in range(K,-1,-1):
    # TODO Calculate the derivatives of the loss with respect to the biases at layer from all_dl_df[layer]. (eq 7.22)
    # NOTE!  To take a copy of matrix X, use Z=np.array(X)
    # REPLACE THIS LINE
    all_dl_dbiases[layer] = np.zeros_like(all_biases[layer])

    # TODO Calculate the derivatives of the loss with respect to the weights at layer from all_dl_df[layer] and all_h[layer] (eq 7.23)
    # Don't forget to use np.matmul
    # REPLACE THIS LINE
    all_dl_dweights[layer] = np.zeros_like(all_weights[layer])

    # TODO: calculate the derivatives of the loss with respect to the activations from weight and derivatives of next preactivations (second part of last line of eq 7.25)
    # REPLACE THIS LINE
    all_dl_dh[layer] = np.zeros_like(all_h[layer])


    if layer > 0:
      # TODO Calculate the derivatives of the loss with respect to the pre-activation f (use derivative of ReLu function, first part of last line of eq. 7.25)
      # REPLACE THIS LINE
      all_dl_df[layer-1] = np.zeros_like(all_f[layer-1])

  # We return the derivatives with respect to the activations and pre-activations too -- you'll need these in Part 2
  return all_dl_dweights, all_dl_dbiases, all_dl_dh, all_dl_df

In [ ]:
all_dl_dweights, all_dl_dbiases, all_dl_dh, all_dl_df = backward_pass(all_weights, all_biases, all_f, all_h, y)

In [ ]:
np.set_printoptions(precision=3)
# Make space for derivatives computed by finite differences
all_dl_dweights_fd = [None] * (K+1)
all_dl_dbiases_fd = [None] * (K+1)

# Let's test if we have the derivatives right using finite differences
delta_fd = 0.000001

# Test the dervatives of the bias vectors
for layer in range(K+1):
  dl_dbias  = np.zeros_like(all_dl_dbiases[layer])
  # For every element in the bias
  for row in range(all_biases[layer].shape[0]):
    # Take copy of biases  We'll change one element each time
    all_biases_copy = [np.array(x) for x in all_biases]
    all_biases_copy[layer][row] += delta_fd
    network_output_1, *_ = compute_network_output(net_input, all_weights, all_biases_copy)
    network_output_2, *_ = compute_network_output(net_input, all_weights, all_biases)
    dl_dbias[row] = (least_squares_loss(network_output_1, y) - least_squares_loss(network_output_2,y))/delta_fd
  all_dl_dbiases_fd[layer] = np.array(dl_dbias)
  print("-----------------------------------------------")
  print("Bias %d, derivatives from backprop:"%(layer))
  print(all_dl_dbiases[layer])
  print("Bias %d, derivatives from finite differences"%(layer))
  print(all_dl_dbiases_fd[layer])
  if np.allclose(all_dl_dbiases_fd[layer],all_dl_dbiases[layer],rtol=1e-05, atol=1e-08, equal_nan=False):
    print("Success!  Derivatives match.")
  else:
    print("Failure!  Derivatives different.")



# Test the derivatives of the weights matrices
for layer in range(K+1):
  dl_dweight  = np.zeros_like(all_dl_dweights[layer])
  # For every element in the bias
  for row in range(all_weights[layer].shape[0]):
    for col in range(all_weights[layer].shape[1]):
      # Take copy of biases  We'll change one element each time
      all_weights_copy = [np.array(x) for x in all_weights]
      all_weights_copy[layer][row][col] += delta_fd
      network_output_1, *_ = compute_network_output(net_input, all_weights_copy, all_biases)
      network_output_2, *_ = compute_network_output(net_input, all_weights, all_biases)
      dl_dweight[row][col] = (least_squares_loss(network_output_1, y) - least_squares_loss(network_output_2,y))/delta_fd
  all_dl_dweights_fd[layer] = np.array(dl_dweight)
  print("-----------------------------------------------")
  print("Weight %d, derivatives from backprop:"%(layer))
  print(all_dl_dweights[layer])
  print("Weight %d, derivatives from finite differences"%(layer))
  print(all_dl_dweights_fd[layer])
  if np.allclose(all_dl_dweights_fd[layer],all_dl_dweights[layer],rtol=1e-05, atol=1e-08, equal_nan=False):
    print("Success!  Derivatives match.")
  else:
    print("Failure!  Derivatives different.")

## **Training a network with your backpropagation code**

Your derivatives match the finite differences, so now let's put them to work and actually train a network.

We'll use the same 30-point dataset from Lab 3. In Lab 3 we fit a Gabor model with just two parameters, and we had to work out its derivatives by hand. This time we'll fit a deep ReLU network with hundreds of parameters, and **your** `compute_network_output()` and `backward_pass()` functions will compute all of the gradients.

There's nothing to write in this section. Just run the cells.

In [ ]:
# The same 30 pairs {x_i, y_i} from Lab 3
data = np.array([[-1.920e+00,-1.422e+01,1.490e+00,-1.940e+00,-2.389e+00,-5.090e+00,
                 -8.861e+00,3.578e+00,-6.010e+00,-6.995e+00,3.634e+00,8.743e-01,
                 -1.096e+01,4.073e-01,-9.467e+00,8.560e+00,1.062e+01,-1.729e-01,
                  1.040e+01,-1.261e+01,1.574e-01,-1.304e+01,-2.156e+00,-1.210e+01,
                 -1.119e+01,2.902e+00,-8.220e+00,-1.179e+01,-8.391e+00,-4.505e+00],
                  [-1.051e+00,-2.482e-02,8.896e-01,-4.943e-01,-9.371e-01,4.306e-01,
                  9.577e-03,-7.944e-02 ,1.624e-01,-2.682e-01,-3.129e-01,8.303e-01,
                  -2.365e-02,5.098e-01,-2.777e-01,3.367e-01,1.927e-01,-2.222e-01,
                  6.352e-02,6.888e-03,3.224e-02,1.091e-02,-5.706e-01,-5.258e-02,
                  -3.666e-02,1.709e-01,-4.805e-02,2.008e-01,-1.904e-01,5.952e-01]])

# The x values run from -15 to 15. We'll divide them by 15 so the network sees inputs in [-1, 1],
# which makes training much better behaved. (We'll undo this when we plot.)
x_scale = 15.0
train_x = data[0:1,:] / x_scale
train_y = data[1:2,:]

We need a fresh network to train. We'll use 3 hidden layers of 20 units each. The biases start at zero, and the random weights are scaled by $\sqrt{2/D_{in}}$, where $D_{in}$ is the number of inputs to that layer. You'll see why that scaling matters in Part 2.

In [ ]:
def init_network(K, D, D_i=1, D_o=1):
  # Set seed so we always get the same random numbers
  np.random.seed(1)
  layer_sizes = [D_i] + [D] * K + [D_o]
  all_weights = [None] * (K+1)
  all_biases = [None] * (K+1)
  for layer in range(K+1):
    fan_in = layer_sizes[layer]
    all_weights[layer] = np.random.normal(size=(layer_sizes[layer+1], fan_in)) * np.sqrt(2.0 / fan_in)
    all_biases[layer] = np.zeros((layer_sizes[layer+1], 1))
  return all_weights, all_biases

# Draw the data and the function computed by the network
def draw_network_model(ax, all_weights, all_biases, title=None):
  x_model = np.arange(-15, 15, 0.1)
  y_model, *_ = compute_network_output(x_model[np.newaxis,:] / x_scale, all_weights, all_biases)
  ax.plot(data[0,:], data[1,:], 'bo')
  ax.plot(x_model, y_model[0,:], 'm-')
  ax.set_xlim([-15,15]); ax.set_ylim([-1,1])
  ax.set_xlabel('x'); ax.set_ylabel('y')
  if title is not None:
    ax.set_title(title)

Here is one step of stochastic gradient descent. Your `backward_pass()` handles one training example at a time, so for each batch we call it once per example, add up the gradients, and divide by the batch size to get the average. Then we take a step downhill:

\begin{equation}
\boldsymbol\Omega_k \leftarrow \boldsymbol\Omega_k - \alpha\frac{\partial L}{\partial \boldsymbol\Omega_k}, \qquad \boldsymbol\beta_k \leftarrow \boldsymbol\beta_k - \alpha\frac{\partial L}{\partial \boldsymbol\beta_k}
\end{equation}

In [ ]:
def sgd_step(all_weights, all_biases, batch_x, batch_y, alpha):
  K = len(all_weights) - 1
  batch_size = batch_x.shape[1]
  # Start with zero gradients for every parameter
  sum_dl_dweights = [np.zeros_like(w) for w in all_weights]
  sum_dl_dbiases = [np.zeros_like(b) for b in all_biases]

  for i in range(batch_size):
    x_i = batch_x[:, i:i+1]
    y_i = batch_y[:, i:i+1]
    # Forward pass and backward pass -- these are YOUR functions from above
    net_output, all_f, all_h = compute_network_output(x_i, all_weights, all_biases)
    all_dl_dweights, all_dl_dbiases, *_ = backward_pass(all_weights, all_biases, all_f, all_h, y_i)
    for layer in range(K+1):
      sum_dl_dweights[layer] += all_dl_dweights[layer]
      sum_dl_dbiases[layer] += all_dl_dbiases[layer]

  # Update every parameter using the average gradient over the batch
  for layer in range(K+1):
    all_weights[layer] = all_weights[layer] - alpha * sum_dl_dweights[layer] / batch_size
    all_biases[layer] = all_biases[layer] - alpha * sum_dl_dbiases[layer] / batch_size
  return all_weights, all_biases

Now let's train. Each **epoch** is one pass through all 30 data points in random batches of 5. This should take a few seconds.

In [ ]:
# Network and training settings
K_train = 3          # Number of hidden layers
D_train = 20         # Number of hidden units per layer
alpha = 0.05         # Learning rate
batch_size = 5
n_epochs = 2000
snapshot_epochs = [0, 50, 500, n_epochs]

train_weights, train_biases = init_network(K_train, D_train)
n_params = sum(w.size for w in train_weights) + sum(b.size for b in train_biases)
print("Number of parameters = %d"%(n_params))

n_data = train_x.shape[1]
losses = []
fig, axes = plt.subplots(1, len(snapshot_epochs), figsize=(16,3.5))
c_snapshot = 0

for epoch in range(n_epochs + 1):
  # Record the loss over the whole training set
  net_output, *_ = compute_network_output(train_x, train_weights, train_biases)
  losses.append(least_squares_loss(net_output, train_y))
  if epoch == snapshot_epochs[c_snapshot]:
    draw_network_model(axes[c_snapshot], train_weights, train_biases, "Epoch %d, loss = %3.3f"%(epoch, losses[-1]))
    c_snapshot += 1
  if epoch % 250 == 0:
    print("Epoch %4d, loss = %3.3f"%(epoch, losses[-1]))
  if epoch == n_epochs:
    break

  # Shuffle the data, then take one SGD step per batch
  shuffled = np.random.permutation(n_data)
  for start in range(0, n_data, batch_size):
    batch = shuffled[start:start+batch_size]
    train_weights, train_biases = sgd_step(train_weights, train_biases, train_x[:,batch], train_y[:,batch], alpha)

plt.tight_layout()
plt.show()

# Plot the loss over training
fig, ax = plt.subplots()
ax.plot(losses)
ax.set_xlabel('Epoch'); ax.set_ylabel('Training loss')
ax.set_yscale('log')
plt.show()

Every gradient used in that training run came from your `backward_pass()` function. This is what PyTorch and other deep learning libraries do under the hood, just with automatic differentiation instead of hand-written derivatives.

**Optional things to try:** change `D_train`, `K_train`, `alpha`, or `n_epochs` and re-run the training cell. What happens if the learning rate is too large? How does the network's fit compare to the Gabor model from Lab 3, and is fitting the training data this closely a good thing?

# **Part 2: Initialization**

Now we'll explore weight initialization in deep neural networks, as described in section 7.5 of the book.

We'll reuse the functions you wrote in Part 1: `ReLU()`, `compute_network_output()`, `least_squares_loss()`, `d_loss_d_output()`, `indicator_function()` and `backward_pass()`. **Before going on, make sure all of the derivatives in Part 1 matched the finite differences.** If they didn't, fix your code first, because everything below depends on it.

The one new thing we need is a way to build networks of different sizes, with the weights drawn from a normal distribution with variance $\sigma^2_\Omega$ and the biases set to zero.

In [ ]:
def init_params(K, D, sigma_sq_omega):
  # Set seed so we always get the same random numbers
  np.random.seed(0)

  # Input layer
  D_i = 1
  # Output layer
  D_o = 1

  # Make empty lists
  all_weights = [None] * (K+1)
  all_biases = [None] * (K+1)

  # Create input and output layers
  all_weights[0] = np.random.normal(size=(D, D_i))*np.sqrt(sigma_sq_omega)
  all_weights[-1] = np.random.normal(size=(D_o, D)) * np.sqrt(sigma_sq_omega)
  all_biases[0] = np.zeros((D,1))
  all_biases[-1]= np.zeros((D_o,1))

  # Create intermediate layers
  for layer in range(1,K):
    all_weights[layer] = np.random.normal(size=(D,D))*np.sqrt(sigma_sq_omega)
    all_biases[layer] = np.zeros((D,1))

  return all_weights, all_biases

Now let's investigate how the size of the outputs vary as we change the initialization variance:


In [ ]:
# Number of layers
K = 5
# Number of neurons per layer
D = 8
# Input layer
D_i = 1
# Output layer
D_o = 1
# Set variance of initial weights to 1
sigma_sq_omega = 1.0
# Initialize parameters
all_weights, all_biases = init_params(K,D,sigma_sq_omega)

n_data = 1000
data_in = np.random.normal(size=(1,n_data))
net_output, all_f, all_h = compute_network_output(data_in, all_weights, all_biases)

for layer in range(1,K+1):
  print("Layer %d, std of hidden units = %3.3f"%(layer, np.std(all_h[layer])))

In [ ]:
# You can see that the values of the hidden units are increasing on average (the variance is across all hidden units at the layer
# and the 1000 training examples

# TODO
# Change this to 50 layers with 80 hidden units per layer

# TODO
# Now experiment with sigma_sq_omega to try to stop the variance of the forward computation exploding

Now let's look at what happens to the magnitude of the gradients on the way back. We'll use the least squares loss and your `backward_pass()` function from Part 1.

In [ ]:
# Number of layers
K = 5
# Number of neurons per layer
D = 8
# Input layer
D_i = 1
# Output layer
D_o = 1
# Set variance of initial weights to 1
sigma_sq_omega = 1.0
# Initialize parameters
all_weights, all_biases = init_params(K,D,sigma_sq_omega)

# For simplicity we'll just consider the gradients of the weights and biases between the first and last hidden layer
n_data = 100
aggregate_dl_df = [None] * (K+1)
for layer in range(1,K):
  # These 3D arrays will store the gradients for every data point
  aggregate_dl_df[layer] = np.zeros((D,n_data))


# We'll have to compute the derivatives of the parameters for each data point separately
for c_data in range(n_data):
  data_in = np.random.normal(size=(1,1))
  y = np.zeros((1,1))
  net_output, all_f, all_h = compute_network_output(data_in, all_weights, all_biases)
  all_dl_dweights, all_dl_dbiases, all_dl_dh, all_dl_df = backward_pass(all_weights, all_biases, all_f, all_h, y)
  for layer in range(1,K):
    aggregate_dl_df[layer][:,c_data] = np.squeeze(all_dl_df[layer])

for layer in reversed(range(1,K)):
  print("Layer %d, std of dl_dh = %3.3f"%(layer, np.std(aggregate_dl_df[layer].ravel())))


In [ ]:
# You can see that the gradients of the hidden units are increasing on average (the standard deviation is across all hidden units at the layer
# and the 100 training examples

# TODO
# Change this to 50 layers with 80 hidden units per layer

# TODO
# Now experiment with sigma_sq_omega to try to stop the variance of the gradients exploding
